# Maritime Perception MVP — Sea / Land / Vessel Segmentation

Goal: a fast, validation-level demo for semantic segmentation of maritime scenes (water / land / sky / obstacles) using the **MaSTr1325** dataset, with a pretrained backbone (transfer learning) for rapid training on a Colab T4 GPU.

**Notebook workflow:**
1. Environment setup
2. Download MaSTr1325 dataset
3. Dataset / DataLoader
4. Model (DeepLabV3+ with pretrained ResNet34 backbone)
5. Training loop
6. Evaluation (IoU per class)
7. Inference + visualization (overlay on image/video)
8. Model export for demo app (Gradio)

> Note: MaSTr1325 is downloaded from the official source (https://box.vicos.si/borja/viamaro/index.html ili GitHub repo `bborja/mastr1325_dataset`). If the link is unavailable, upload a smaller subset manually to Colab.

## 1. Setup

In [ ]:
!pip install -q segmentation-models-pytorch albumentations gradio opencv-python-headless

import os
import torch
import torch.nn as nn
import numpy as np
import cv2
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
assert device.type == 'cuda', 'Go to Runtime > Change runtime type > GPU and re-run.'

## 2. Download MaSTr1325 dataset

MaSTr1325 contains 1325 annotated maritime scene images with 4 labels: **obstacles/environment (0), water (1), sky (2), ignore/unknown (4)**. We use official direct download links.

In [ ]:
!mkdir -p /content/data/mastr1325/all_images /content/data/mastr1325/all_masks

# Official direct download links (from https://box.vicos.si/borja/viamaro/index.html):
!wget -q --show-progress -O /content/data/mastr1325_images.zip https://box.vicos.si/borja/mastr1325_dataset/MaSTr1325_images_512x384.zip
!wget -q --show-progress -O /content/data/mastr1325_masks.zip https://box.vicos.si/borja/mastr1325_dataset/MaSTr1325_masks_512x384.zip

!unzip -q /content/data/mastr1325_images.zip -d /content/data/mastr1325/all_images
!unzip -q /content/data/mastr1325_masks.zip -d /content/data/mastr1325/all_masks

print('Number of images:', len(os.listdir('/content/data/mastr1325/all_images')))
print('Number of masks:', len(os.listdir('/content/data/mastr1325/all_masks')))

### 2b. Train/val split
The dataset ships as a single flat folder — we create a 90/10 train/val split.

In [ ]:
import shutil
from sklearn.model_selection import train_test_split

# Dataset ships as a flat folder — we create a 90/10 train/val split
all_img_dir = '/content/data/mastr1325/all_images'
all_mask_dir = '/content/data/mastr1325/all_masks'

filenames = sorted([f for f in os.listdir(all_img_dir) if f.lower().endswith(('.jpg', '.png'))])
train_files, val_files = train_test_split(filenames, test_size=0.1, random_state=42)

for split_name, files in [('train', train_files), ('val', val_files)]:
    img_out = f'/content/data/mastr1325/{split_name}/images'
    mask_out = f'/content/data/mastr1325/{split_name}/masks'
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(mask_out, exist_ok=True)
    for fname in files:
        shutil.copy(os.path.join(all_img_dir, fname), os.path.join(img_out, fname))
        mask_name = os.path.splitext(fname)[0] + 'm.png'  # mask filenames have 'm' appended (e.g. 0001m.png)
        mask_src = os.path.join(all_mask_dir, mask_name)
        if os.path.exists(mask_src):
            shutil.copy(mask_src, os.path.join(mask_out, mask_name))

print(f'Train: {len(train_files)} images | Val: {len(val_files)} images')

**Size note:** The images zip is large (hundreds of MB), masks zip is small. If the Colab connection drops, just re-run the cell (wget nastavlja gdje je stao samo ako dodaš `-c`, ili jednostavno ponovo pokreni od početka — fajlovi nisu preveliki za Colab disk).

**IMPORTANT — actual MaSTr1325 classes (3 classes + ignore region, NOT 4):**
- `0` = obstacles and environment (land, vessels, all obstacles — **all in one class**)
- `1` = water
- `2` = sky
- `4` = ignore/unknown region (class boundaries — excluded from training loss, **not** a prediction class)

Vessels are **not a separate class** in this dataset — they fall under "obstacles". To separately detect vessels (vessel vs. land vs. other obstacle), add a YOLO detection model on top of the segmentation mask: trening dodatnog detekcionog modela (npr. YOLO) koji radi NA TOP segmentacione maske, fokusiran samo na regione klasifikovane kao "obstacle".

## 3. Dataset Classes and Augmentations

In [ ]:
# Actual MaSTr1325 classes: 0=obstacles/environment, 1=water, 2=sky, 4=ignore/unknown
# Vessels are NOT a separate class — they fall under 'obstacles' along with land and all other obstacles.
# Ignore region (4) is remapped to 255 and excluded from the loss function (ignore_index), as recommended by dataset authors.
NUM_CLASSES = 3  # 0=obstacle/environment, 1=water, 2=sky
CLASS_NAMES = ['obstacle_environment', 'water', 'sky']
IGNORE_INDEX = 255
CLASS_COLORS = {
    0: (0, 0, 255),     # obstacle/environment (land+vessels+all obstacles) - red (BGR for cv2)
    1: (255, 120, 0),   # water - blue/orange
    2: (200, 200, 200), # sky - grey
}

def remap_mask(mask):
    """Remap original MaSTr1325 label 4 (ignore) -> 255 (ignore_index); everything else unchanged."""
    mask = mask.copy()
    mask[mask == 4] = IGNORE_INDEX
    return mask

IMG_SIZE = 384

train_tfms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_tfms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

class MaritimeSegDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.filenames = sorted(os.listdir(images_dir))
        self.transform = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        img_path = os.path.join(self.images_dir, fname)
        mask_path = os.path.join(self.masks_dir, os.path.splitext(fname)[0] + 'm.png')  # 'm' appended in mask filename

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = remap_mask(mask)  # remap 4 -> 255 (ignore_index)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        return image, mask.long()

# Adjust paths after download (Step 2)
TRAIN_IMAGES = '/content/data/mastr1325/train/images'
TRAIN_MASKS = '/content/data/mastr1325/train/masks'
VAL_IMAGES = '/content/data/mastr1325/val/images'
VAL_MASKS = '/content/data/mastr1325/val/masks'

# train_ds = MaritimeSegDataset(TRAIN_IMAGES, TRAIN_MASKS, transform=train_tfms)
# val_ds = MaritimeSegDataset(VAL_IMAGES, VAL_MASKS, transform=val_tfms)
# train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2)
# val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)
print('Dataset class ready. Uncomment the lines above once paths are correct.')

## 4. Model — DeepLabV3+ with pretrained ResNet34

In [ ]:
model = smp.DeepLabV3Plus(
    encoder_name='resnet34',
    encoder_weights='imagenet',
    in_channels=3,
    classes=NUM_CLASSES,
).to(device)

loss_fn = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)  # boundary pixels (4->255) are excluded from the loss
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

print('Model parameters:', sum(p.numel() for p in model.parameters()) / 1e6, 'M')

## 5. Training loop

In [ ]:
def iou_per_class(preds, targets, num_classes, ignore_index=255):
    ious = []
    preds = preds.view(-1)
    targets = targets.view(-1)
    valid = targets != ignore_index
    preds = preds[valid]
    targets = targets[valid]
    for c in range(num_classes):
        pred_c = (preds == c)
        target_c = (targets == c)
        intersection = (pred_c & target_c).sum().item()
        union = (pred_c | target_c).sum().item()
        ious.append(intersection / union if union > 0 else float('nan'))
    return ious

def train_one_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total_loss = 0
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def validate(model, loader, loss_fn, num_classes):
    model.eval()
    total_loss = 0
    all_ious = []
    with torch.no_grad():
        for images, masks in loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            loss = loss_fn(outputs, masks)
            total_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            all_ious.append(iou_per_class(preds, masks, num_classes))
    mean_ious = np.nanmean(np.array(all_ious), axis=0)
    return total_loss / len(loader), mean_ious

# EPOCHS = 30
# best_miou = 0
# for epoch in range(EPOCHS):
#     train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn)
#     val_loss, ious = validate(model, val_loader, loss_fn, NUM_CLASSES)
#     scheduler.step()
#     miou = np.nanmean(ious)
#     print(f'Epoch {epoch+1}/{EPOCHS} | train_loss={train_loss:.4f} val_loss={val_loss:.4f} mIoU={miou:.4f}')
#     print('  IoU po klasi:', dict(zip(CLASS_NAMES, [round(x,3) for x in ious])))
#     if miou > best_miou:
#         best_miou = miou
#         torch.save(model.state_dict(), '/content/best_model.pth')
#         print('  -> sačuvan novi najbolji model')
print('Training loop definisan. Otkomentariši petlju kad su DataLoader-i spremni.')

## 6. Inference + Visualization (overlay)

In [ ]:
def predict_and_overlay(model, image_bgr, alpha=0.5):
    """Takes a BGR image (cv2 format), returns overlay with color-coded segmentation masks."""
    model.eval()
    orig_h, orig_w = image_bgr.shape[:2]
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    augmented = val_tfms(image=image_rgb, mask=np.zeros(image_rgb.shape[:2], dtype=np.uint8))
    input_tensor = augmented['image'].unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_tensor)
        pred_mask = torch.argmax(output, dim=1).squeeze(0).cpu().numpy().astype(np.uint8)

    pred_mask_resized = cv2.resize(pred_mask, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)

    color_mask = np.zeros_like(image_bgr)
    for class_id, color in CLASS_COLORS.items():
        color_mask[pred_mask_resized == class_id] = color

    overlay = cv2.addWeighted(image_bgr, 1 - alpha, color_mask, alpha, 0)
    return overlay, pred_mask_resized

# Test on a single image:
# test_img = cv2.imread('/content/data/mastr1325/val/images/SAMPLE.jpg')
# overlay, mask = predict_and_overlay(model, test_img)
# plt.figure(figsize=(12,6))
# plt.subplot(1,2,1); plt.imshow(cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB)); plt.title('Original'); plt.axis('off')
# plt.subplot(1,2,2); plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)); plt.title('Segmentation'); plt.axis('off')
# plt.show()
print('Inference function ready.')

## 7. Video Inference (optional — for demo recording)

In [ ]:
def process_video(model, input_path, output_path, alpha=0.5, max_frames=None):
    cap = cv2.VideoCapture(input_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (w, h))

    frame_count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        overlay, _ = predict_and_overlay(model, frame, alpha=alpha)
        out.write(overlay)
        frame_count += 1
        if max_frames and frame_count >= max_frames:
            break

    cap.release()
    out.release()
    print(f'Sačuvano: {output_path} ({frame_count} frejmova)')

# process_video(model, '/content/test_video.mp4', '/content/output_demo.mp4', max_frames=300)
print('Video inference function ready — use to record a demo video.')

## 8. Gradio Demo App
Run this cell at the end — it gives you a web interface link where you can upload an image and see the result live.

In [ ]:
import gradio as gr

def gradio_predict(input_image):
    image_bgr = cv2.cvtColor(np.array(input_image), cv2.COLOR_RGB2BGR)
    overlay, _ = predict_and_overlay(model, image_bgr)
    overlay_rgb = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)
    return overlay_rgb

demo = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Image(type='numpy', label='Maritime scene (upload)'),
    outputs=gr.Image(type='numpy', label='Segmentation: obstacle/water/sky'),
    title='Maritime Perception MVP — Sea/Land/Vessel Segmentation',
    description='Maritime autonomy perception model demo. Upload any maritime scene image.',
)

# demo.launch(share=True)  # share=True daje javni link koji možeš poslati drugima
print('Gradio app defined. Run demo.launch(share=True) once the model is trained.')

## Next Steps After This Notebook

1. Load the MaSTr1325 dataset (Step 2) and uncomment the DataLoaders (Step 3)
2. Run training (Step 5) — on a T4 GPU, ~30 epochs on 1325 images typically takes 20-40 min
3. Record a short demo video with the overlay (Step 7) — use for pitch deck
4. Launch the Gradio app with `share=True` and share the link as a live prototype
5. (Next phase) Add vessel detection (YOLO) on top of the segmentation mask — the 'characteristic model' layer

## Findings — Generalization Testing (Post-MVP Analysis)

After training (mIoU = 0.989 on MaSTr1325 val split) and testing on out-of-distribution images, a classic **domain shift** problem was observed — the model performs excellently on data similar to the training distribution, but does not generalize well to visually different scenarios. Detailed findings per test image:

### Test 1 — MaSTr1325-style image (eye-level, USV camera)
**Result: near-perfect.** Segmentation of water, sky, and vessels (including fine details such as flags on masts) is highly accurate and cleanly follows object boundaries.

### Test 2 — Aerial / drone shot of a marina (top-down view)
**Errors:**
- Large open water is correctly classified (water)
- Sky and background land/trees misclassified as **obstacle** instead of **sky**
- Small bright/reflective water patches between boat hulls misclassified as **sky** instead of **water**

**Root cause:** MaSTr1325 consists exclusively of eye-level USV footage where sky always dominates the upper frame as a large continuous region. From an aerial angle, the sky shrinks to a thin distant strip the model cannot recognize as its learned 'sky' pattern, while bright reflective water patches resemble the sky's brightness signature from training.

### Test 3 — Eye-level photo with clear, deep blue sky (no clouds)
**Error:** The entire clear blue sky is classified as **water** instead of **sky**; vessels and actual water are correctly segmented.

**Root cause:** MaSTr1325 training images predominantly feature hazy, overcast, whitish-grey skies (typical of low-altitude maritime photography in humid conditions). A clear, saturated blue sky closely resembles calm water in the training distribution — the model defaults to 'water' when it sees that color because water is the dominant class with that signature.

### Key pattern across all three tests

| Class | Robustness to distribution shift |
|---|---|
| **Vessels / obstacle** | Consistently accurate across all three tests — the safety-critical class is also the most robust |
| **Water** | Reliable for large continuous regions; fails on edge cases (small reflective patches, unusual camera angles) |
| **Sky** | Least robust class — sensitive to atmospheric conditions (clear vs. hazy) and camera angle (eye-level vs. aerial) |

### Implications for real-world deployment

These findings directly relate to the challenges described in maritime AI perception roles — where off-the-shelf models are known to fail in dynamic maritime conditions. Specifically:

1. **Training data diversity across camera angles and atmospheric conditions** is critical for robust generalization — directly relevant to 'design-of-experiment methods for data collection and annotation' mentioned in maritime AI engineering roles
2. **Prioritizing safety-critical classes** — a system that reliably detects vessels/obstacles (critical for collision avoidance) but errs on sky (less critical) is an acceptable trade-off
3. **Next step to improve generalization**: expand the training dataset with images from multiple sources (drone footage, varied atmospheric conditions, clear vs. overcast skies), or apply domain adaptation techniques before relying on the model outside the MaSTr1325 distribution.
